# dpa-adapt Quick Start Tutorial

Fine-tune a pre-trained DPA-3 model for molecular property prediction — from data preparation to model deployment in under 10 minutes on CPU.

<a href="https://nb.bohrium.dp.tech/detail/52879961357" target="_blank"><img src="https://cdn.dp.tech/bohrium/web/static/images/open-in-bohrium.svg" alt="Open In Bohrium"/></a>

<div style="color: black; background-color: #E8F5E9; border: 1px solid #A5D6A7; border-radius: 10px; margin-bottom: 1.5rem; padding: 1.5rem; font-family: inherit;">
    <div style="border-bottom: 1px dashed #A5D6A7; padding-bottom: 1rem; margin-bottom: 1rem;">
        <span style="background-color: #C8E6C9; color: #1B5E20; padding: 0.2rem 0.5rem; border-radius: 4px; font-size: 0.75rem; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">Quick Start</span>
        <p style="margin: 0.5rem 0 0 0; font-size: 1.05rem; line-height: 1.6; font-weight: bold; color: #1B5E20;">
            Adapt pre-trained DPA models to your property prediction tasks. Go from a handful of labeled molecules to a deployable predictor in minutes.
        </p>
    </div>
    <p style="margin: 0; font-size: 0.92rem; line-height: 1.8; color: #455A64;">
        Training accurate machine-learning potentials from scratch requires massive DFT datasets and significant compute. Pre-trained DPA models solve the data problem: they have already learned rich representations of atomic interactions across millions of structures spanning the periodic table. <b style="color: #1B5E20;">dpa-adapt</b> lets you transfer that knowledge to your specific task — whether it's predicting HOMO–LUMO gaps, band gaps, formation energies, or any molecular or materials property — with as few as dozens of labeled examples.
    </p>
</div>

## Task

> **Mastering the dpa-adapt workflow: from a pre-trained DPA checkpoint and labeled data to a frozen, deployable property predictor.**

By the end of this tutorial, you will be able to:

* Format Data: Convert raw molecular data into the standard deepmd/npy format required by dpa-adapt.
* Select Strategy: Load a pre-trained DPA backbone and navigate the trade-offs between four adaptive modes (frozen_sklearn, linear_probe, finetune, mft) based on dataset size and hardware availability.
* Train & Evaluate: Fit an property predictor and benchmark its accuracy using standard regression metrics (MAE, RMSE, $R^2$).
* Freeze & Deploy: Compile the adapted pipeline into a self-contained .pth bundle for zero-dependency downstream deployment.


```{contents} Table of Contents
:depth: 3
```

## Background

### What is dpa-adapt?

**dpa-adapt** is a scikit-learn-style Python package for adapting pre-trained DPA models to downstream property prediction. The acronym **ADAPT** stands for *Atomistic DPA Adaptation for Property Tasks*.

The package appears under three names, each serving a different context:

| Name | Context | Example |
|------|---------|---------|
| `dpa-adapt` | PyPI package, pip install, docs | `pip install deepmd-kit[dpa-adapt]` |
| `dpaad` | CLI command | `dpaad fit --train-data ./data ...` |
| `dpa_adapt` | Python import | `from dpa_adapt import DPAFineTuner` |

### Fine-tuning strategies

dpa-adapt offers four strategies. All share the same pre-trained DPA backbone; they differ in how much of it gets updated:

| Strategy | Core Mechanism | Target Data Size | Hardware Regime | Primary Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **`frozen_sklearn`** | Frozen backbone + Scikit-learn regressor | Small ($< 1\text{k}$) | CPU Only | Ultra-fast benchmarking & prototyping |
| **`linear_probe`** | Frozen backbone + Gradient-descent linear head | Medium ($1\text{k} - 10\text{k}$) | CPU / GPU | Balanced efficiency for linear properties |
| **`finetune`** | End-to-end full parameter fine-tuning | Large ($> 10\text{k}$) | GPU Required | Maximum accuracy on massive datasets |
| **`mft`** | Multi-task co-training (Property + Force Field) | Small / Low-data | GPU Required | Mitigating representation collapse |

In this tutorial we use **`frozen_sklearn`** — it runs on CPU, needs no GPU, and delivers useful accuracy on small datasets. We'll predict the **HOMO–LUMO gap** of small organic molecules from the QM9 (GDB9) dataset, using a DPA-3.1 model pre-trained on the Drugs domain.

For the full API reference, see the [dpa-adapt documentation](https://docs.deepmodeling.com/projects/deepmd/en/master/dpa_adapt/index.html).

## Practice

### Prerequisites and Setup

Before we begin, ensure you have the required packages installed:

```bash
pip install deepmd-kit[dpa-adapt]
```

You will also need a DPA pre-trained checkpoint. This demo uses **DPA-3.1-3M** with `model_branch="Domains_Drug"`. You can download pre-trained models from [AIS Square](https://www.aissquare.com) or the DeepModeling release page.


In [ ]:
import os
from pathlib import Path
import numpy as np

# Force CPU mode — avoids device-mismatch errors when the checkpoint
# was saved with CUDA tensors. Remove this line if you have a GPU and
# want to use it (may require additional setup).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

# Resolve paths relative to this notebook's location
HERE = Path().resolve()
DATA_DIR = HERE / "data"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

print(f"Working directory : {HERE}")
print(f"Training data     : {TRAIN_DIR}")
print(f"Test data         : {TEST_DIR}")

In [ ]:
# Define the dataset URL and the paths
dataset_url = "https://bohrium-api.dp.tech/ds-dl/dpa-adapt-quickstart-v1.zip"  # TODO: update when uploaded
zip_file_name = "dpa-adapt-quickstart-v1.zip"
dataset_directory = "dpa-adapt-quickstart"
local_zip_path = f"/personal/{zip_file_name}"
extract_path = "/personal/"

# Check if the dataset directory exists to avoid re-downloading and re-extracting
if not os.path.isdir(f"{extract_path}{dataset_directory}"):
    # Download and extract if not exists
    if not os.path.isfile(local_zip_path):
        print("Downloading dataset...")
        !wget -q -O {local_zip_path} {dataset_url}

    print("Extracting dataset...")
    !unzip -q -n {local_zip_path} -d {extract_path}
else:
    print("Dataset is already downloaded and extracted.")

# Change the current working directory
os.chdir(f"{extract_path}")
print(f"Current path is: {os.getcwd()}")

### Data Preparation

We have prepared a subset of 50 molecules from the QM9 (GDB9) dataset, already converted to the `deepmd/npy` format required by dpa-adapt. The data is split into 40 training molecules and 10 test molecules.

Let's take a look at the data directory structure:

In [ ]:
! tree data/ -L 1

The `data/` folder contains two subdirectories:

- `train/` — 40 molecular systems (`sys_0000` through `sys_0039`) for training
- `test/` — 10 molecular systems (`sys_0000` through `sys_0009`) for evaluation

Each `sys_*/` sub-directory is a self-contained system in DeePMD-kit's compressed NumPy format. Let's inspect one training system:

In [ ]:
! tree data/train/sys_0000/

Each system directory contains:

- **`set.000/`** — a directory holding the compressed NumPy arrays for coordinates, forces, energies, cells, and (optionally) labels such as `gap.npy`.
- **`type.raw`** — a file listing the atomic type indices (integers) for each atom in the system.
- **`type_map.raw`** — a file mapping type indices to chemical element symbols.

Let's look at the type information for a sample molecule:

In [ ]:
# Show the atom types and type mapping for a sample system
print("=== type.raw ===")
! cat data/train/sys_0000/type.raw
print("\n=== type_map.raw ===")
! cat data/train/sys_0000/type_map.raw

The type map tells us this molecule contains Hydrogen (H), Carbon (C), Nitrogen (N), Oxygen (O), and Fluorine (F) atoms. The `type.raw` file encodes each atom as its index into this map.

The ground-truth target (the HOMO–LUMO gap in eV) is encapsulated within set.000/gap.npy, which dpa-adapt automatically fetches via the target_key="gap" directive.

> **Note:** The pre-processed data was generated from raw GDB9 using `scripts/prepare_data.py`. If you want to use your own molecules, you can follow the same pattern — convert each molecule to a `deepmd/npy` system and place your target values in `set.000/<your_key>.npy`.

More detailed documentation on using dpdata for data conversion can be found in the [DeePMD-kit documentation](https://docs.deepmodeling.com/projects/deepmd/en/master/data/data-conv.html).

### Step 1 — Load the Pre-trained DPA Model

The `DPAFineTuner` class is the main entry point. It loads a pre-trained DPA checkpoint and configures it for fine-tuning. The **`frozen_sklearn`** strategy freezes the DPA backbone, extracts atomic descriptors, and fits a scikit-learn regressor on top — no GPU needed.

The key parameters are:

| Parameter | Description | Our value |
|-----------|-------------|-----------|
| `pretrained` | Model name (auto-downloaded) or path to a pre-trained DPA checkpoint (`.pt` file) | `"DPA-3.1-3M"` |
| `model_branch` | Which domain the pre-trained model was trained on | `"Domains_Drug"` |
| `strategy` | Fine-tuning strategy: `frozen_sklearn`, `linear_probe`, `finetune`, or `mft` | `"frozen_sklearn"` |
| `predictor` | Type of scikit-learn predictor for `frozen_sklearn` strategy (`"linear"` for Ridge, `"rf"` for Random Forest) | `"linear"` |
| `pooling` | How to aggregate per-atom descriptors into a molecule-level vector (`"mean"`, `"sum"`, `"max"`) | `"mean"` |
| `seed` | Random seed for reproducibility | `42` |

For the `finetune` and `mft` strategies, additional parameters like `learning_rate`, `max_steps`, `batch_size`, and `loss_function` control the neural network training loop — these are documented in the [dpa-adapt API reference](https://docs.deepmodeling.com/projects/deepmd/en/master/dpa_adapt/).

In [ ]:
from dpa_adapt import DPAFineTuner

model = DPAFineTuner(
    pretrained="DPA-3.1-3M",       # auto-downloaded from AIS Square
    model_branch="Domains_Drug",
    strategy="frozen_sklearn",
    predictor="linear",
    pooling="mean",
    seed=42,
)
print(f"Strategy: {model.strategy}")
print(f"Model branch: {model.model_branch}")

### Step 2 — Fit the Model

The `fit()` method takes a glob pattern that matches system directories. With `frozen_sklearn`, it:

1. **Extracts descriptors** — runs each molecule through the frozen DPA backbone to produce fixed-size descriptor vectors.
2. **Fits a regressor** — trains a scikit-learn Ridge (or Random Forest) regressor on the descriptor → label pairs.

The `target_key="gap"` argument tells the method to look for `set.000/gap.npy` inside each system directory to read the label.

> **⏱️ Expected time:** ~30 seconds for 40 molecules on CPU.

In [ ]:
model.fit(train_data=str(TRAIN_DIR) + "/*", target_key="gap")
print("Training complete!")

### Step 3 — Evaluate on the Held-out Test Set

The `evaluate()` method runs the fine-tuned model on unseen test data and returns a set of regression metrics:

- **MAE** (Mean Absolute Error) — average absolute deviation from the true value, in eV.
- **RMSE** (Root Mean Square Error) — square root of the average squared error, penalizing large errors more heavily.
- **R²** (coefficient of determination) — how well the predictions correlate with the true values (1.0 is perfect).

We also get back the raw predictions, which we can use for visualization.

In [ ]:
metrics = model.evaluate(data=str(TEST_DIR) + "/*")
print(f"MAE  : {metrics.mae:.4f} eV")
print(f"RMSE : {metrics.rmse:.4f} eV")
print(f"R²   : {metrics.r2:.4f}")
print(f"N    : {metrics.predictions.shape[0]}")

### Visualize Predictions

We can visualize the correlation between the predicted values and the true (DFT) values. A good model should have points clustered around the diagonal line.

In [ ]:
import matplotlib.pyplot as plt

# Load true labels for the test set
true_gaps = np.load(DATA_DIR / "test_labels.npy")
pred_gaps = metrics.predictions

# Scatter plot
plt.figure(figsize=(6, 5))
plt.scatter(true_gaps, pred_gaps, alpha=0.7, edgecolors="k", linewidths=0.5)

# Diagonal reference line
x_range = np.linspace(min(true_gaps), max(true_gaps), 100)
plt.plot(x_range, x_range, "r--", linewidth=0.75, label="Perfect prediction")

plt.xlabel("True HOMO–LUMO gap (eV)")
plt.ylabel("Predicted HOMO–LUMO gap (eV)")
plt.title(f"dpa-adapt — frozen_sklearn\nMAE = {metrics.mae:.4f} eV, R² = {metrics.r2:.4f}")
plt.legend()
plt.tight_layout()
plt.show()

### Step 4 — Freeze and Reload the Model

Freezing saves the fine-tuned model as a self-contained bundle (`.pth` file) that can be loaded with the lightweight `DPAPredictor` — no training dependencies required. This is the preferred format for deployment and sharing.

The frozen bundle includes:
- The model weights (or, for `frozen_sklearn`, the fitted sklearn pipeline)
- Metadata about the pooling method, type map, and model configuration
- Everything needed to run inference on new molecules

In [ ]:
# Freeze the fine-tuned model
frozen_path = "frozen_model.pth"
model.freeze(frozen_path)
print(f"Model frozen to: {frozen_path}")

# Reload with the lightweight predictor
from dpa_adapt import DPAPredictor

predictor = DPAPredictor(frozen_path)
result = predictor.predict(str(TEST_DIR) + "/*")
print(f"Predictions shape     : {result.predictions.shape}")
print(f"First 5 predictions   : {result.predictions[:5].round(4)}")
print(f"Reloaded MAE          : {np.abs(result.predictions - true_gaps).mean():.4f} eV")

### Trying Other Strategies

The `frozen_sklearn` strategy we used above is the fastest path to a working model. When you have more data or need higher accuracy, switch strategies by changing a single parameter:

**`linear_probe`** — neural head on frozen descriptors, trained with gradient descent (no GPU):
```python
model = DPAFineTuner(
    pretrained="DPA-3.1-3M",
    model_branch="Domains_Drug",
    strategy="linear_probe",
    pooling="mean",
    learning_rate=0.001,
    max_steps=5000,
    seed=42,
)
```

**`finetune`** — update the full DPA model end-to-end (GPU recommended):
```python
model = DPAFineTuner(
    pretrained="DPA-3.1-3M",
    model_branch="Domains_Drug",
    strategy="finetune",
    pooling="mean",
    learning_rate=0.001,
    max_steps=100000,
    batch_size="auto:512",
    seed=42,
)
```

**`mft`** — multi-task fine-tuning: the property head trains alongside an auxiliary force-field head to prevent representation collapse on small datasets:
```python
model = DPAFineTuner(
    pretrained="DPA-3.1-3M",
    model_branch="Domains_Drug",
    strategy="mft",
    pooling="mean",
    aux_branch="MP_traj_v024_alldata_mixu",
    aux_prob=0.5,
    seed=42,
)
```

The `fit()` / `evaluate()` / `freeze()` workflow is identical across all strategies — only the constructor changes. See the [dpa-adapt documentation](https://docs.deepmodeling.com/projects/deepmd/en/master/dpa_adapt/) for the full parameter reference.

## Next Steps

Congratulations! You've completed the dpa-adapt (ADAPT) quick start tutorial. Here's what you can explore next:

- **Try other strategies** — Experiment with `linear_probe`, `finetune`, and `mft` to see how accuracy improves with more powerful fine-tuning approaches.
- **Use your own data** — Replace `TRAIN_DIR` / `TEST_DIR` with your own `deepmd/npy` directories and set `target_key` to match your label key. Use `scripts/prepare_data.py` as a reference for converting your molecular data. You can also use the `dpaad data convert` CLI for automatic format detection.
- **Tune hyperparameters** — Adjust `pooling` (`"mean"`, `"sum"`, `"max"`), `predictor` (`"linear"`, `"rf"`), and for neural network strategies, `learning_rate`, `batch_size`, and `max_steps`.
- **Explore multi-task learning** — The `mft` strategy can leverage auxiliary data from large datasets like MP_traj to improve data efficiency on small downstream datasets.
- **Read the full documentation** — Visit the [dpa-adapt documentation](https://docs.deepmodeling.com/projects/deepmd/en/master/dpa_adapt/) for API references, advanced configuration, and more examples.
- **Check the DeePMD-kit quick start** — If you're also interested in training Deep Potential models from scratch for molecular dynamics, see the [DeePMD-kit Quick Start Tutorial](../getting-started/quick_start.ipynb).

---

<div style="color: black; background-color: #E8F5E9; border: 1px solid #A5D6A7; border-radius: 10px; margin-bottom: 1.5rem; padding: 1.5rem; font-family: inherit;">
    <div style="border-bottom: 1px dashed #A5D6A7; padding-bottom: 1rem; margin-bottom: 1rem;">
        <span style="background-color: #C8E6C9; color: #1B5E20; padding: 0.2rem 0.5rem; border-radius: 4px; font-size: 0.75rem; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">🎉 Mission Accomplished!</span>
        <p style="margin: 0.5rem 0 0 0; font-size: 1.05rem; line-height: 1.6; font-weight: bold; color: #1B5E20;">
            With just a few lines of code, you've successfully fine-tuned a pre-trained DPA model, evaluated its accuracy, and frozen it for seamless deployment.
        </p>
    </div>
    <p style="margin: 0; font-size: 0.92rem; line-height: 1.6; color: #455A64;">
        ⚡ <b>High Efficiency:</b> The entire pipeline was executed fully on a standard <b>CPU</b> and completed in <b>under 10 minutes</b>, demonstrating the low-data and low-compute advantages of <b>dpa-adapt</b>.
    </p>
</div>